In [1]:
import jax.numpy as jnp
import numpy as np
import jax
import matplotlib.pyplot as plt

In [343]:
def _kalman_update(
    mean_prev: jnp.ndarray,
    cov_prev: jnp.ndarray,
    obs: jnp.ndarray,
    transition_matrix: jnp.ndarray,
    process_cov: jnp.ndarray,
    measurement_matrix: jnp.ndarray,
    measurement_cov: jnp.ndarray,
):
    """

    Parameters
    ----------
    mean_prev : jnp.ndarray, shape (n_states,)
        Previous state mean.
    cov_prev : jnp.ndarray, shape (n_states, n_states)
        Previous state covariance.
    obs : jnp.ndarray, shape (n_obs_dim,)
        Data observation.
    transition_matrix : jnp.ndarray, shape (n_states, n_states)
        State transition matrix.
    process_cov : jnp.ndarray, shape (n_states, n_states)
        State noise covariance.
    measurement_matrix : jnp.ndarray, shape (n_obs_dim, n_states)
        Maps the observation to the state
    measurement_cov : jnp.ndarray, shape (n_obs_dim, n_obs_dim)
        Observation noise covariance.
    """
    # jax.debug.print("mean_prev: {}", mean_prev)
    # jax.debug.print("cov_prev: {}", cov_prev)
    # jax.debug.print("obs: {}", obs)
    # jax.debug.print("transition_matrix: {}", transition_matrix)
    # jax.debug.print("process_cov: {}", process_cov)
    # jax.debug.print("measurement_matrix: {}", measurement_matrix)
    # jax.debug.print("measurement_cov: {}", measurement_cov)

    # One step prediction
    one_step_mean = transition_matrix @ mean_prev
    one_step_cov = transition_matrix @ cov_prev @ transition_matrix.T + process_cov

    # Measurement update
    obs_mean = measurement_matrix @ one_step_mean
    # obs_cross_cov = one_step_cov @ measurement_matrix.T
    obs_cov = measurement_matrix @ one_step_cov @ measurement_matrix.T + measurement_cov

    residual_error = obs - obs_mean  # innovation
    kalman_gain = jax.scipy.linalg.solve(
        obs_cov, measurement_matrix @ one_step_cov, assume_a="pos"
    ).T

    posterior_mean = one_step_mean + kalman_gain @ residual_error
    posterior_cov = one_step_cov - kalman_gain @ obs_cov @ kalman_gain.T

    marginal_log_likelihood = jax.scipy.stats.multivariate_normal.logpdf(
        x=residual_error, mean=jnp.zeros_like(residual_error), cov=obs_cov
    )
    # cross_variance = (jnp.identity(mean_prev.shape[0]) - kalman_gain @ measurement_matrix) @ transition_matrix @ cov_prev

    return posterior_mean, posterior_cov, kalman_gain, marginal_log_likelihood


@jax.jit
def collapse_MoG(
    conditional_means_x: jnp.ndarray,
    conditional_cov: jnp.ndarray,
    mixing_weights: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Collapse a mixture of Gaussians.

    Parameters
    ----------
    conditional_means_x : jnp.ndarray, shape (n_dims, n_discrete_states)
        E[X | S = j]
    conditional_cov : jnp.ndarray, shape (n_dims, n_dims, n_discrete_states)
        Cov[X | S = j]
    mixing_weights : jnp.ndarray, shape (n_discrete_states,)
        P[S = j]

    Returns
    -------
    unconditional_mean_x : jnp.ndarray, shape (n_dims,)
        E[X]
    unconditional_cov_x : jnp.ndarray, shape (n_dims, n_dims)
        Cov[X]
    """
    unconditional_mean_x = conditional_means_x @ mixing_weights  # E[X]
    diff_x = conditional_means_x - unconditional_mean_x[:, None]

    unconditional_cov_xx = (
        conditional_cov @ mixing_weights + (diff_x * mixing_weights) @ diff_x.T
    )  # E[XX]

    return unconditional_mean_x, unconditional_cov_xx


def collapse_MoG_cross(
    conditional_means_x: jnp.ndarray,
    conditional_means_y: jnp.ndarray,
    conditional_cross_cov: jnp.ndarray,
    mixing_weights: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Collapse a mixture of Gaussians.

    Parameters
    ----------
    conditional_means_x : jnp.ndarray, shape (n_dims, n_discrete_states)
        E[X | S = j]
    conditional_means_y : jnp.ndarray, shape (n_dims, n_discrete_states)
        E[Y | S = j]
    conditional_cross_cov : jnp.ndarray, shape (n_dims, n_dims, n_discrete_states)
        E[X,Y | S = j]
    mixing_weights : jnp.ndarray, shape (n_discrete_states,)
        P[S = j]

    Returns
    -------
    unconditional_mean_x : jnp.ndarray, shape (n_dims,)
        E[X]
    unconditional_mean_y : jnp.ndarray, shape (n_dims,)
        E[Y]
    unconditional_cov_xy : jnp.ndarray, shape (n_dims, n_dims)
        E[X,Y]
    """

    unconditional_mean_x = conditional_means_x @ mixing_weights  # E[X]
    unconditional_mean_y = conditional_means_y @ mixing_weights  # E[Y]

    diff_x = conditional_means_x - unconditional_mean_x[:, None]
    diff_y = conditional_means_y - unconditional_mean_y[:, None]

    unconditional_cov_xy = (
        conditional_cross_cov @ mixing_weights + (diff_x * mixing_weights) @ diff_y.T
    )  # E[XY]

    return unconditional_mean_x, unconditional_mean_y, unconditional_cov_xy


collapse_MoG_vmap = jax.vmap(collapse_MoG, in_axes=(2, 3, 1), out_axes=(1, 2))


def switching_kalman_filter(
    init_mean: jnp.ndarray,
    init_cov: jnp.ndarray,
    init_discrete_state_prob: jnp.ndarray,
    obs: jnp.ndarray,
    discrete_transition_matrix: jnp.ndarray,
    continuous_transition_matrix: jnp.ndarray,
    process_cov: jnp.ndarray,
    measurement_matrix: jnp.ndarray,
    measurement_cov: jnp.ndarray,
):
    """Switching Kalman filter for a linear Gaussian state space model with discrete states.

    Parameters
    ----------
    init_mean : jnp.ndarray, shape (n_cont_states,)
        Initial value of the continuous latent state
    init_cov : jnp.ndarray, shape (n_cont_states, n_cont_states)
        Initial covariance of the continuous latent state
    init_discrete_state_prob : jnp.ndarray, shape (n_discrete_states,)
        Initial probability of the discrete states
    obs : jnp.ndarray, shape (n_time, n_obs_dim)
        Observations
    discrete_transition_matrix : jnp.ndarray, shape (n_discrete_states, n_discrete_states)
        Transition matrix for the discrete states
    continuous_transition_matrix : jnp.ndarray, shape (n_cont_states, n_cont_states, n_discrete_states)
        Transition matrix for the continuous states
    process_cov : jnp.ndarray, shape (n_cont_states, n_cont_states, n_discrete_states)
        Process noise covariance
    measurement_matrix : jnp.ndarray, shape (n_obs_dims, n_cont_states, n_discrete_states)
        Map observations to the continuous states
    measurement_cov : jnp.ndarray, shape (n_obs_dims, n_obs_dims, n_discrete_states)
        Measurement variance

    Returns
    -------
    _type_
        _description_
    """

    n_discrete_states = discrete_transition_matrix.shape[0]
    n_cont_states = continuous_transition_matrix.shape[0]

    def _step(carry, obs_t):
        mean_prev, cov_prev, discrete_state_prob_prev = carry

        # mean_prev, shape (n_cont_states, n_discrete_states)
        # cov_prev, shape (n_cont_states, n_cont_states, n_discrete_states)

        posterior_mean = jnp.zeros(
            (n_discrete_states, n_discrete_states, n_cont_states)
        )
        posterior_cov = jnp.zeros(
            (n_discrete_states, n_discrete_states, n_cont_states, n_cont_states)
        )
        kalman_gain = jnp.zeros(
            (n_discrete_states, n_discrete_states, n_cont_states, n_cont_states)
        )
        marginal_log_likelihood = jnp.zeros((n_discrete_states, n_discrete_states))

        # Kalman update for each pair of discrete states
        # vmap twice over the discrete states
        for state_j in range(n_discrete_states):
            for state_i in range(n_discrete_states):
                (
                    posterior_mean_ij,
                    posterior_cov_ij,
                    kalman_gain_ij,
                    marginal_log_likelihood_ij,
                ) = _kalman_update(
                    mean_prev[:, state_i],
                    cov_prev[:, :, state_i],
                    obs_t,
                    continuous_transition_matrix[:, :, state_j],
                    process_cov[:, :, state_j],
                    measurement_matrix[:, :, state_j],
                    measurement_cov[:, :, state_j],
                )
                posterior_mean.at[state_i, state_j].set(posterior_mean_ij)
                posterior_cov.at[state_i, state_j].set(posterior_cov_ij)
                kalman_gain.at[state_i, state_j].set(kalman_gain_ij)
                marginal_log_likelihood.at[state_i, state_j].set(
                    marginal_log_likelihood_ij
                )

        # Need to make sure the likelihood is normalized to max 1 for numerical stability
        conditional_marginal_likelihood = jnp.exp(
            marginal_log_likelihood - jnp.max(marginal_log_likelihood)
        )

        # Compute the probability of the discrete states and the weights
        # joint prob between time steps
        joint_discrete_state_prob = (
            conditional_marginal_likelihood
            * discrete_transition_matrix
            * discrete_state_prob_prev[:, None]
        ) # shape (n_discrete_states, n_discrete_states) p(S_t, S_{t-1})
        joint_discrete_state_prob /= jnp.sum(joint_discrete_state_prob)

        discrete_state_prob = jnp.sum(
            joint_discrete_state_prob, axis=0
        )  # shape (n_discrete_states,) p(S_t)
        discrete_state_weights = (
            joint_discrete_state_prob / discrete_state_prob[None, :]
        )  # joint / marginal = conditional p(S_t | S_{t-1})

        # Collapse n_discrete_states x n_discrete_states Gaussians to n_discrete_states Gaussians
        collapsed_mean, collapsed_cov = collapse_MoG_vmap(
            posterior_mean,
            posterior_cov,
            discrete_state_weights,
        )

        return (collapsed_mean, collapsed_cov, discrete_state_prob), (
            collapsed_mean,
            collapsed_cov,
            discrete_state_prob,
        )

    (_, _, _), (filter_mean, filter_cov, filter_discrete_state_prob) = jax.lax.scan(
        _step,
        (init_mean, init_cov, init_discrete_state_prob),
        obs,
    )

    return filter_mean, filter_cov, filter_discrete_state_prob

In [344]:
n_obs_dims = 2
n_cont_states = 3
n_discrete_states = 4

mean_prev = np.random.rand(n_cont_states, n_discrete_states)
cov_prev = np.random.rand(n_cont_states, n_cont_states, n_discrete_states)
obs_t = np.random.rand(n_obs_dims)
continuous_transition_matrix = np.random.rand(
    n_cont_states, n_cont_states, n_discrete_states
)
process_cov = np.random.rand(n_cont_states, n_cont_states, n_discrete_states)
measurement_matrix = np.random.rand(n_obs_dims, n_cont_states, n_discrete_states)
measurement_cov = np.random.rand(n_obs_dims, n_obs_dims, n_discrete_states)

posterior_mean = jnp.zeros((n_discrete_states, n_discrete_states, n_cont_states))
posterior_cov = jnp.zeros(
    (n_discrete_states, n_discrete_states, n_cont_states, n_cont_states)
)
kalman_gain = jnp.zeros(
    (n_discrete_states, n_discrete_states, n_cont_states, n_obs_dims)
)
marginal_log_likelihood = jnp.zeros((n_discrete_states, n_discrete_states))

for state_j in range(n_discrete_states):
    for state_i in range(n_discrete_states):
        (
            posterior_mean_ij,
            posterior_cov_ij,
            kalman_gain_ij,
            marginal_log_likelihood_ij,
        ) = _kalman_update(
            mean_prev[:, state_i],
            cov_prev[:, :, state_i],
            obs_t,
            continuous_transition_matrix[:, :, state_j],
            process_cov[:, :, state_j],
            measurement_matrix[:, :, state_j],
            measurement_cov[:, :, state_j],
        )
        posterior_mean = posterior_mean.at[state_i, state_j, :].set(posterior_mean_ij)
        posterior_cov = posterior_cov.at[state_i, state_j].set(posterior_cov_ij)
        kalman_gain = kalman_gain.at[state_i, state_j].set(kalman_gain_ij)
        marginal_log_likelihood = marginal_log_likelihood.at[state_i, state_j].set(
            marginal_log_likelihood_ij
        )

posterior_mean[0, 0]

Array([0.36794567, 0.10151142, 0.5460142 ], dtype=float32)

In [219]:
(
    posterior_mean_ij,
    posterior_cov_ij,
    kalman_gain_ij,
    marginal_log_likelihood_ij,
) = _kalman_update(
    mean_prev[:, state_i],
    cov_prev[:, :, state_i],
    obs_t,
    continuous_transition_matrix[:, :, state_j],
    process_cov[:, :, state_j],
    measurement_matrix[:, :, state_j],
    measurement_cov[:, :, state_j],
)

mean_prev: [0.9944677  0.9139661  0.07947573]


In [226]:
from functools import partial

vmap_posterior_mean = jnp.zeros((n_discrete_states, n_discrete_states, n_cont_states))
vmap_posterior_cov = jnp.zeros(
    (n_discrete_states, n_discrete_states, n_cont_states, n_cont_states)
)
vmap_kalman_gain = jnp.zeros(
    (n_discrete_states, n_discrete_states, n_cont_states, n_obs_dims)
)
vmap_marginal_log_likelihood = jnp.zeros((n_discrete_states, n_discrete_states))

for state_j in range(n_discrete_states):
    vmapped_kalman_update_partial = partial(
        _kalman_update,
        obs=obs_t,
        transition_matrix=continuous_transition_matrix[:, :, state_j],
        process_cov=process_cov[:, :, state_j],
        measurement_matrix=measurement_matrix[:, :, state_j],
        measurement_cov=measurement_cov[:, :, state_j],
    )
    vmapped_kalman_update = jax.vmap(vmapped_kalman_update_partial, in_axes=(1, 2))
    (
        posterior_mean_j,
        posterior_cov_j,
        kalman_gain_j,
        marginal_log_likelihood_j,
    ) = vmapped_kalman_update(mean_prev, cov_prev)

    vmap_posterior_mean = vmap_posterior_mean.at[:, state_j, :].set(posterior_mean_j)
    vmap_posterior_cov = vmap_posterior_cov.at[:, state_j].set(posterior_cov_j)
    vmap_kalman_gain = vmap_kalman_gain.at[:, state_j].set(kalman_gain_j)
    vmap_marginal_log_likelihood = vmap_marginal_log_likelihood.at[:, state_j].set(
        marginal_log_likelihood_j
    )

(
    np.allclose(posterior_mean, vmap_posterior_mean, equal_nan=True),
    np.allclose(posterior_cov, vmap_posterior_cov, equal_nan=True),
    np.allclose(kalman_gain, vmap_kalman_gain, equal_nan=True),
    np.allclose(marginal_log_likelihood, vmap_marginal_log_likelihood, equal_nan=True),
)

mean_prev shape: (Array(3, dtype=int32, weak_type=True),)
mean_prev shape: (Array(3, dtype=int32, weak_type=True),)
mean_prev shape: (Array(3, dtype=int32, weak_type=True),)
mean_prev shape: (Array(3, dtype=int32, weak_type=True),)


(False, False, False, True)

In [262]:
state_j = 0

for state_i in range(n_discrete_states):
    (
        posterior_mean_ij,
        posterior_cov_ij,
        kalman_gain_ij,
        marginal_log_likelihood_ij,
    ) = _kalman_update(
        mean_prev[:, state_i],
        cov_prev[:, :, state_i],
        obs_t,
        continuous_transition_matrix[:, :, state_j],
        process_cov[:, :, state_j],
        measurement_matrix[:, :, state_j],
        measurement_cov[:, :, state_j],
    )

    print(posterior_mean_ij)

[0.74201363 0.73311335 0.38051522]
[0.7724651  0.8536643  0.44095182]
[0.580965   0.6679565  0.36005235]
[0.6528832  0.747861   0.21477833]


In [300]:
vmapped_kalman_update_partial = partial(
    _kalman_update,
    obs=obs_t,
    transition_matrix=continuous_transition_matrix[:, :, state_j],
    process_cov=process_cov[:, :, state_j],
    measurement_matrix=measurement_matrix[:, :, state_j],
    measurement_cov=measurement_cov[:, :, state_j],
)
vmapped_kalman_update = jax.vmap(vmapped_kalman_update_partial, in_axes=(1, 2))
vmapped_kalman_update(mean_prev, cov_prev)[0]

Array([[0.8950453 , 0.8171046 , 0.3276408 ],
       [0.9776645 , 0.96515584, 0.3950364 ],
       [0.6443633 , 0.6831831 , 0.28193644],
       [0.69324875, 0.73901653, 0.17345399]], dtype=float32)

In [252]:
vmapped_kalman_update = jax.vmap(
    jax.vmap(_kalman_update, in_axes=(None, None, None, 2, 2, 2, 2)),
    in_axes=(1, 2, None, None, None, None, None),
)

(
    vmap_posterior_mean,
    vmap_posterior_cov,
    vmap_kalman_gain,
    vmap_marginal_log_likelihood,
) = vmapped_kalman_update(
    mean_prev,
    cov_prev,
    obs_t,
    continuous_transition_matrix,
    process_cov,
    measurement_matrix,
    measurement_cov,
)

(
    np.allclose(posterior_mean, vmap_posterior_mean, equal_nan=True),
    np.allclose(posterior_cov, vmap_posterior_cov, equal_nan=True),
    np.allclose(kalman_gain, vmap_kalman_gain, equal_nan=True),
    np.allclose(marginal_log_likelihood, vmap_marginal_log_likelihood, equal_nan=True),
)

(False, False, False, True)